# OpenEvals: Evaluating LLM Outputs

In [ ]:
%pip install openevals rouge-score datasets nltk python-dotenv -q

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # loads ANTHROPIC_API_KEY from .env

from datasets import load_dataset
from openevals.llm import create_llm_as_judge
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
nltk.download('punkt_tab', quiet=True)

## 1. The free-form evaluation problem

In [ ]:
question = "What is the capital of France?"
reference = "Paris"

model_a = "Paris is the capital of France."
model_b = "The capital city of France, a country in Western Europe, is Paris, which has been its capital since the 10th century."

# Both are correct — which one do you pick? How?
print("Model A:", model_a)
print("Model B:", model_b)

## 2. Token-based metrics (with ground truth)

**BLEU** counts n-gram overlaps between output and reference. **ROUGE** measures recall of reference n-grams in the output.  
Both require a ground-truth reference. We use [HLE](https://huggingface.co/datasets/cais/hle) — PhD-level expert questions — as our benchmark.

In [ ]:
# Load text-only, exactMatch examples from HLE
ds = load_dataset("cais/hle", split="test", streaming=True)

examples = []
for sample in ds:
    if len(examples) >= 10:
        break
    if sample["answer_type"] == "exactMatch" and not sample["image"]:
        examples.append({
            "question": sample["question"],
            "reference": sample["answer"],
        })

print(f"Loaded {len(examples)} text-only examples")
print("\nExample question:", examples[0]["question"][:150])
print("Reference answer:", examples[0]["reference"])

In [ ]:
# Simulate a model that just returns the reference (upper bound)
# and one that returns a plausible but slightly different answer
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
smoother = SmoothingFunction().method1

for ex in examples[:3]:
    ref = ex["reference"]
    pred = ref  # perfect prediction
    
    bleu = sentence_bleu([ref.split()], pred.split(), smoothing_function=smoother)
    rouge = scorer.score(ref, pred)["rougeL"].fmeasure
    
    print(f"Ref: {ref!r}")
    print(f"  BLEU={bleu:.3f}  ROUGE-L={rouge:.3f}")
    
    # Now a slightly wrong answer
    pred_wrong = "I don't know"
    bleu_w = sentence_bleu([ref.split()], pred_wrong.split(), smoothing_function=smoother)
    rouge_w = scorer.score(ref, pred_wrong)["rougeL"].fmeasure
    print(f"  Wrong pred BLEU={bleu_w:.3f}  ROUGE-L={rouge_w:.3f}")
    print()

**Key limitation**: BLEU/ROUGE are sensitive to surface form. A semantically correct paraphrase may score near zero.

## 3. LLM-as-a-Judge: anatomy of a good prompt

From the slides, a good judge prompt has four elements:

1. **Task description** — *Your task is to do X*
2. **What you will be given** — *You will be provided with Y*
3. **Evaluation criteria** — *You should evaluate property Z on a scale of 1–5 where 1 means…*
4. **Reasoning steps + output spec** — *Identify first… then verify… output True or False*

We build two variations of a correctness prompt to see how each element changes judge behaviour.

In [ ]:
# V1 — minimal: task description + criteria + output spec
CORRECTNESS_V1 = """\
Your task is to evaluate whether an answer is correct.

You will be provided with a question, a candidate answer, and a reference answer.

An answer is correct if it conveys the same factual information as the reference.
An answer is incorrect if it is factually wrong, misleading, or contradicts the reference.

Output True if the answer is correct, False otherwise.

<input>
{inputs}
</input>

<output>
{outputs}
</output>

<reference_outputs>
{reference_outputs}
</reference_outputs>
"""

# V2 — full anatomy: adds explicit reasoning steps (CoT) following the slide model
CORRECTNESS_V2 = """\
Your task is to evaluate whether an answer is correct.

You will be provided with a question, a candidate answer, and a reference answer.

You should evaluate correctness on a binary scale where:
- True means: the answer is factually accurate and semantically equivalent to the reference \
(minor phrasing differences are acceptable; semantic equivalence is what matters)
- False means: the answer is factually wrong, incomplete in a meaningful way, \
or contradicts the reference

Reasoning steps:
1. Read the reference answer and identify the key facts it asserts.
2. Read the candidate answer and check whether it asserts the same facts.
3. Decide: does the candidate answer convey the same meaning as the reference?
4. Output True or False.

<input>
{inputs}
</input>

<output>
{outputs}
</output>

<reference_outputs>
{reference_outputs}
</reference_outputs>
"""

judge_v1 = create_llm_as_judge(
    prompt=CORRECTNESS_V1,
    model="anthropic:claude-haiku-4-5-20251001",
    feedback_key="correctness",
)

judge_v2 = create_llm_as_judge(
    prompt=CORRECTNESS_V2,
    model="anthropic:claude-haiku-4-5-20251001",
    feedback_key="correctness",
)

## 4. Do the two prompts agree?

The interesting cases are the **borderline** ones — semantically correct paraphrases, partial answers, or minor factual gaps. V1 may be inconsistent here; V2's reasoning steps guide the judge through the decision.

In [ ]:
eval_cases = [
    # Clear correct — both should agree
    {
        "label": "correct paraphrase",
        "inputs":           {"question": "What is the boiling point of water at sea level?"},
        "outputs":          {"answer": "100 degrees Celsius"},
        "reference_outputs":{"answer": "100°C (212°F)"},
    },
    # Clear wrong — both should agree
    {
        "label": "wrong answer",
        "inputs":           {"question": "What is the boiling point of water at sea level?"},
        "outputs":          {"answer": "It depends on altitude."},
        "reference_outputs":{"answer": "100°C (212°F)"},
    },
    # Borderline: partially correct, different phrasing
    {
        "label": "partial / ambiguous",
        "inputs":           {"question": "What is photosynthesis?"},
        "outputs":          {"answer": "A process plants use to make food from sunlight."},
        "reference_outputs":{"answer": "The process by which plants convert CO2 and water into glucose using sunlight, releasing oxygen."},
    },
    # Borderline: correct fact, wrong unit
    {
        "label": "correct fact, wrong units",
        "inputs":           {"question": "What is the boiling point of water at sea level?"},
        "outputs":          {"answer": "212 degrees Fahrenheit"},
        "reference_outputs":{"answer": "100°C"},
    },
]

print(f"{'Case':<25} {'V1':>6} {'V2':>6}")
print("-" * 40)
for case in eval_cases:
    r1 = judge_v1(inputs=case["inputs"], outputs=case["outputs"], reference_outputs=case["reference_outputs"])
    r2 = judge_v2(inputs=case["inputs"], outputs=case["outputs"], reference_outputs=case["reference_outputs"])
    agree = "✓" if r1["score"] == r2["score"] else "✗ disagree"
    print(f"{case['label']:<25} {str(r1['score']):>6} {str(r2['score']):>6}  {agree}")

## 5. Judge biases

LLM judges are not objective. Known biases: positional (prefers first/second option), verbosity (prefers longer answers), self-preference.

In [ ]:
PAIRWISE_PROMPT = """\
Which response better answers the question? Reply with exactly 'A' or 'B'.

Question: {inputs}
Response A: {outputs[a]}
Response B: {outputs[b]}
"""

pairwise_judge = create_llm_as_judge(
    prompt=PAIRWISE_PROMPT,
    model="anthropic:claude-haiku-4-5-20251001",
    feedback_key="winner",
)

question = "Explain what a p-value is."
short_ans = "The probability of observing your data if the null hypothesis is true."
long_ans  = "A p-value is a statistical measure that quantifies the probability of observing results at least as extreme as the ones actually observed in your data, assuming the null hypothesis is true. It ranges from 0 to 1."

# Order 1: short=A, long=B
r1 = pairwise_judge(inputs=question, outputs={"a": short_ans, "b": long_ans})
# Order 2: long=A, short=B  (swapped)
r2 = pairwise_judge(inputs=question, outputs={"a": long_ans, "b": short_ans})

print(f"Short=A, Long=B  → Winner: {r1['score']}")
print(f"Long=A,  Short=B → Winner: {r2['score']}")
print()
print("If the judge is consistent, it should prefer the same answer regardless of order.")
print("If results differ, that is positional bias.")

## 6. Calibrating the judge

Run both prompt versions against a hand-labeled set and compare accuracy — this is how you pick between prompt designs.

In [ ]:
calibration_set = [
    {"inputs": {"question": "What is 2+2?"},                    "outputs": {"answer": "4"},                "reference_outputs": {"answer": "4"},                   "human_label": True},
    {"inputs": {"question": "What is 2+2?"},                    "outputs": {"answer": "5"},                "reference_outputs": {"answer": "4"},                   "human_label": False},
    {"inputs": {"question": "Name the first US president."},    "outputs": {"answer": "George Washington"}, "reference_outputs": {"answer": "George Washington"},   "human_label": True},
    {"inputs": {"question": "Name the first US president."},    "outputs": {"answer": "Abraham Lincoln"},   "reference_outputs": {"answer": "George Washington"},   "human_label": False},
    {"inputs": {"question": "What gas do plants absorb?"},      "outputs": {"answer": "CO2"},               "reference_outputs": {"answer": "Carbon dioxide"},      "human_label": True},
    {"inputs": {"question": "What gas do plants absorb?"},      "outputs": {"answer": "Oxygen"},            "reference_outputs": {"answer": "Carbon dioxide"},      "human_label": False},
    {"inputs": {"question": "What is the speed of light?"},     "outputs": {"answer": "Approximately 3×10⁸ m/s"}, "reference_outputs": {"answer": "299,792,458 m/s"}, "human_label": True},
    {"inputs": {"question": "What is the speed of light?"},     "outputs": {"answer": "Very fast"},         "reference_outputs": {"answer": "299,792,458 m/s"},     "human_label": False},
]

def calibrate(judge, dataset):
    preds, labels = [], []
    for item in dataset:
        result = judge(inputs=item["inputs"], outputs=item["outputs"], reference_outputs=item["reference_outputs"])
        preds.append(result["score"])
        labels.append(item["human_label"])
    tp = sum(p and l for p, l in zip(preds, labels))
    tn = sum(not p and not l for p, l in zip(preds, labels))
    fp = sum(p and not l for p, l in zip(preds, labels))
    fn = sum(not p and l for p, l in zip(preds, labels))
    acc  = (tp + tn) / len(labels)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return {"accuracy": acc, "precision": prec, "recall": rec, "predictions": preds}

results_v1 = calibrate(judge_v1, calibration_set)
results_v2 = calibrate(judge_v2, calibration_set)

print(f"{'Metric':<12} {'V1 (minimal)':>14} {'V2 (CoT)':>12}")
print("-" * 40)
for metric in ["accuracy", "precision", "recall"]:
    print(f"{metric:<12} {results_v1[metric]:>14.2f} {results_v2[metric]:>12.2f}")

print()
print("Human labels:", [item["human_label"] for item in calibration_set])
print("V1 predicted:", results_v1["predictions"])
print("V2 predicted:", results_v2["predictions"])